# Smoke test S_0 → S_1 trên D_dev (Task.md mục 0.1)

Notebook này chạy toàn bộ smoke test trên Google Colab. Hướng dẫn chi tiết: `Instruction_0.1.md` (mục "Chạy trên Colab").

**Trước khi chạy:**
1. `Runtime → Change runtime type → GPU` (T4 miễn phí là đủ; L4/A100 nhanh hơn và dùng bf16 đúng như config).
2. Tải `smoke_bundle.zip` lên Google Drive vào thư mục `MyDrive/DACNTT_smoke/`.
3. Chạy lần lượt từng ô. Mọi kết quả và adapter được ghi thẳng vào Drive, nên nếu Colab bị ngắt thì chạy lại từ đầu: các bước đã xong sẽ được bỏ qua.

**Không dùng D_test ở bất kỳ bước nào.**

In [18]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

name, memory.total [MiB]
Tesla T4, 15360 MiB
torch 2.11.0+cu130 | CUDA True | bf16 True


## 1. Kết nối Drive và giải nén bundle

In [19]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
found = glob.glob('/content/drive/MyDrive/**/smoke_bundle*.zip', recursive=True)
print(found)
assert found, 'Không tìm thấy smoke_bundle.zip trong My Drive'

BUNDLE = found[0]
DRIVE_DIR = os.path.dirname(BUNDLE)
RESULTS = f'{DRIVE_DIR}/results'
ADAPTERS = f'{DRIVE_DIR}/adapters'
WORK = '/content/manual'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(ADAPTERS, exist_ok=True)
print('BUNDLE =', BUNDLE)
print('RESULTS =', RESULTS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['/content/drive/MyDrive/DACNTT/Smoke/smoke_bundle.zip', '/content/drive/MyDrive/DACNTT/Smoke/smoke_bundle (Unzipped Files)/notebooks/smoke_bundle.zip']
BUNDLE = /content/drive/MyDrive/DACNTT/Smoke/smoke_bundle.zip
RESULTS = /content/drive/MyDrive/DACNTT/Smoke/results


In [20]:
!rm -rf {WORK} && mkdir -p {WORK} && unzip -q -o "{BUNDLE}" -d {WORK}
%cd {WORK}
!sha256sum -c MANIFEST.sha256
# kết quả ghi thẳng vào Drive
!mkdir -p reports/task5 && rm -rf reports/task5/smoke_test && ln -s "{RESULTS}" reports/task5/smoke_test
!ls -la reports/task5/

/content/manual
scripts/answer_utils.py: OK
scripts/student_common.py: OK
scripts/generate_teacher_trajectories_batch.py: OK
scripts/eval_student.py: OK
scripts/select_smoke_train_set.py: OK
scripts/train_round.py: OK
scripts/compute_nll.py: OK
scripts/analyze_smoke_test.py: OK
config/experiment_config.json: OK
splits/d_dev_ids.txt: OK
data/raw/gsm8k_train.jsonl: OK
trajectory/pool/batch_100_teacher_verified.jsonl: OK
notebooks/smoke_test_colab.ipynb: OK
total 8
drwxr-xr-x 2 root root 4096 Oct 19 10:08 .
drwxr-xr-x 3 root root 4096 Oct 19 10:08 ..
lrwxrwxrwx 1 root root   43 Oct 19 10:08 smoke_test -> /content/drive/MyDrive/DACNTT/Smoke/results


## 2. Cài thư viện

Ghim đúng phiên bản trong `requirements-lock.txt` của repo cho các thư viện ảnh hưởng tới kết quả. Giữ nguyên torch có sẵn của Colab; phiên bản thực tế được ghi vào `environment_colab.txt`.

In [21]:
!pip install -q transformers==5.15.0 peft==0.20.0 accelerate==1.14.0 scipy==1.17.1
!python -c "import torch, transformers, peft, scipy; print(torch.__version__, transformers.__version__, peft.__version__, scipy.__version__)"
!pip freeze > "{RESULTS}/environment_colab.txt" && nvidia-smi >> "{RESULTS}/environment_colab.txt"
# Nếu tải model bị treo, bỏ dấu # ở dòng dưới rồi chạy lại các bước sau:
# os.environ['HF_HUB_DISABLE_XET'] = '1'

2.11.0+cu130 5.15.0 0.20.0 1.17.1


## 3. Tham số theo GPU

- GPU hỗ trợ bf16 (L4, A100): chạy bf16 như config.
- T4: không có bf16, nên script tự dùng fp32 (tránh fp16 vì Qwen2.5 có thể tràn số). Chậm hơn nhưng an toàn.

Nếu gặp lỗi hết bộ nhớ (CUDA out of memory), giảm `EVAL_BS` hoặc `MICRO_BS` rồi chạy lại ô bị lỗi.

In [22]:
BF16 = torch.cuda.is_bf16_supported()
EVAL_BS = 32 if BF16 else 16      # batch khi sinh lời giải
MICRO_BS = 4 if BF16 else 2       # micro-batch khi huấn luyện (effective batch = 8 theo config)
NLL_BS = 8 if BF16 else 4
SEEDS = [13, 42]
print(dict(BF16=BF16, EVAL_BS=EVAL_BS, MICRO_BS=MICRO_BS, NLL_BS=NLL_BS, SEEDS=SEEDS))

{'BF16': True, 'EVAL_BS': 32, 'MICRO_BS': 4, 'NLL_BS': 8, 'SEEDS': [13, 42]}


## 4. Bước A: đánh giá S_0 trên D_dev (500 câu)

Nếu bị ngắt giữa chừng, chạy lại ô này: các câu đã có kết quả sẽ được bỏ qua.

In [23]:
!python scripts/eval_student.py --tag S0 --batch-size {EVAL_BS} --out reports/task5/smoke_test/eval_S0_dev.jsonl

[S0] 500 problems, 500 already done, 0 to run, dtype=torch.bfloat16
Loading weights: 100% 338/338 [00:00<00:00, 5199.23it/s]
[S0] accuracy 0.7120  format 0.9160  mean_len 238.5  -> reports/task5/smoke_test/eval_S0_dev.summary.json


Kiểm tra nhanh trước khi đi tiếp: accuracy S_0 và **tỉ lệ đúng định dạng** (`format_compliance`). Nếu tỉ lệ đúng định dạng thấp, phần lớn thay đổi sau huấn luyện có thể chỉ là học định dạng: so với `lenient_accuracy_exploratory`.

In [24]:
import json
s = json.load(open('reports/task5/smoke_test/eval_S0_dev.summary.json'))
{k: s[k] for k in ['accuracy', 'format_compliance', 'lenient_accuracy_exploratory', 'mean_output_tokens', 'n_hit_max_tokens', 'elapsed_seconds_this_run', 'peak_gpu_memory_gb', 'dtype']}

{'accuracy': 0.712,
 'format_compliance': 0.916,
 'lenient_accuracy_exploratory': 0.762,
 'mean_output_tokens': 238.538,
 'n_hit_max_tokens': 9,
 'elapsed_seconds_this_run': 0.0,
 'peak_gpu_memory_gb': 2.88,
 'dtype': 'bfloat16'}

## 5. Bước B: chọn 200 trajectory cho mỗi seed

Chỉ trong smoke test: bỏ `n_cap = 1` vì pool sơ bộ chỉ có 98 câu.

In [25]:
for seed in SEEDS:
    !python scripts/select_smoke_train_set.py --seed {seed}

seed 13: 200 trajectories over 88 problems (max 4 per problem) -> /content/manual/reports/task5/smoke_test/train_set_seed13.json
seed 42: 200 trajectories over 92 problems (max 5 per problem) -> /content/manual/reports/task5/smoke_test/train_set_seed42.json


In [26]:
!pip uninstall -y torchao

## 6. Bước B–C: huấn luyện 1 vòng rồi đánh giá S_1, cho từng seed

In [27]:
import os, shutil
for seed in SEEDS:
    adapter = f'{ADAPTERS}/student_ckpt_001_smoke_seed{seed}'
    if os.path.exists(f'{adapter}/train_log.json'):
        print(f'seed {seed}: adapter đã có, bỏ qua huấn luyện')
    else:
        !python scripts/train_round.py --seed {seed} --micro-batch {MICRO_BS} --train-set reports/task5/smoke_test/train_set_seed{seed}.json --adapter-dir "{adapter}"
    shutil.copy(f'{adapter}/train_log.json', f'reports/task5/smoke_test/train_log_seed{seed}.json')
    !python scripts/eval_student.py --tag S1_seed{seed} --adapter "{adapter}" --batch-size {EVAL_BS} --out reports/task5/smoke_test/eval_S1_dev_seed{seed}.jsonl

Loading weights: 100% 338/338 [00:00<00:00, 4913.68it/s]
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
step   1  loss 0.2910  grad_norm 0.394  tokens 1655  17s
step   2  loss 0.2041  grad_norm 0.310  tokens 1836  35s
step   3  loss 0.2601  grad_norm 0.321  tokens 1853  58s
step   4  loss 0.1823  grad_norm 0.238  tokens 2084  81s
step   5  loss 0.1899  grad_norm 0.255  tokens 1658  100s
step   6  loss 0.2002  grad_norm 0.263  tokens 1851  123s
step   7  loss 0.2211  grad_norm 0.255  tokens 1954  146s
step   8  loss 0.2331  grad_norm 0.316  tokens 1560  163s
step   9  loss 0.1828  grad_norm 0.258  tokens 2058  186s
step  10  loss 0.1759  grad_norm 0.344  tokens 1499  203s
step  11  loss 0.2272  grad_norm 0.325  tokens 1943  225s
step  12  loss 0.1969  grad_norm 0.286  tokens 1765  244s
step  13  loss 0.2440  grad_norm 0.316  tokens 2055  267s
step  14  loss 0.2034  grad_norm 0.255  tokens 1861  292s
step  15  loss 0.2395  grad_nor

## 7. Bước D: NLL của pool (480 trajectory) tại S_0 và S_1

In [28]:
import os
if not os.path.exists('reports/task5/smoke_test/nll_pool_S0.summary.json'):
    !python scripts/compute_nll.py --tag S0 --batch-size {NLL_BS} --out reports/task5/smoke_test/nll_pool_S0.jsonl
for seed in SEEDS:
    if not os.path.exists(f'reports/task5/smoke_test/nll_pool_S1_seed{seed}.summary.json'):
        adapter = f'{ADAPTERS}/student_ckpt_001_smoke_seed{seed}'
        !python scripts/compute_nll.py --tag S1_seed{seed} --adapter "{adapter}" --batch-size {NLL_BS} --out reports/task5/smoke_test/nll_pool_S1_seed{seed}.jsonl

Loading weights: 100% 338/338 [00:00<00:00, 5136.84it/s]
[S0] 8/480  6s
[S0] 16/480  13s
[S0] 24/480  22s
[S0] 32/480  31s
[S0] 40/480  39s
[S0] 48/480  45s
[S0] 56/480  52s
[S0] 64/480  57s
[S0] 72/480  64s
[S0] 80/480  71s
[S0] 88/480  77s
[S0] 96/480  83s
[S0] 104/480  90s
[S0] 112/480  98s
[S0] 120/480  106s
[S0] 128/480  110s
[S0] 136/480  117s
[S0] 144/480  125s
[S0] 152/480  132s
[S0] 160/480  137s
[S0] 168/480  143s
[S0] 176/480  149s
[S0] 184/480  156s
[S0] 192/480  161s
[S0] 200/480  166s
[S0] 208/480  172s
[S0] 216/480  177s
[S0] 224/480  183s
[S0] 232/480  189s
[S0] 240/480  195s
[S0] 248/480  202s
[S0] 256/480  210s
[S0] 264/480  217s
[S0] 272/480  225s
[S0] 280/480  231s
[S0] 288/480  235s
[S0] 296/480  244s
[S0] 304/480  250s
[S0] 312/480  255s
[S0] 320/480  259s
[S0] 328/480  264s
[S0] 336/480  272s
[S0] 344/480  277s
[S0] 352/480  284s
[S0] 360/480  290s
[S0] 368/480  297s
[S0] 376/480  304s
[S0] 384/480  312s
[S0] 392/480  317s
[S0] 400/480  323s
[S0] 408/480  329s
[S

## 8. Phân tích và quyết định theo ngưỡng đã chốt

In [29]:
!python scripts/analyze_smoke_test.py --seeds {' '.join(map(str, SEEDS))}

# Smoke test: kết quả phân tích tự động

Ngưỡng: |ΔAcc| ≥ 2% hoặc McNemar p < 0.1.

| Seed | Acc S_0 | Acc S_1 | Δ | sai→đúng | đúng→sai | p | Format S_0→S_1 | Loss đầu→cuối | Quyết định |
|---|---|---|---|---|---|---|---|---|---|
| 13 | 0.712 | 0.830 | +0.118 | 89 | 30 | 5.69e-08 | 0.916→0.996 | 0.225→0.189 | Accuracy tăng rõ, loss giảm: giữ I3/I4/I5 (CẢNH BÁO: tỉ lệ đúng định dạng đổi ≥ 5 điểm %, so với accuracy lỏng trước khi kết luận) |
| 42 | 0.712 | 0.818 | +0.106 | 84 | 31 | 8.03e-07 | 0.916→0.966 | 0.235→0.208 | Accuracy tăng rõ, loss giảm: giữ I3/I4/I5 |

Accuracy lỏng (exploratory, số cuối trong output, bỏ qua định dạng):
- seed 13: 0.762 → 0.830
- seed 42: 0.762 → 0.830

**Quyết định chung:** `keep`
Chênh lệch Δ giữa các seed: 0.012

## NLL (exploratory), seed 13

| Nhóm | n | Spearman ℓ(S_0), ℓ(S_1) | Δℓ trung bình |
|---|---|---|---|
| all | 480 | 0.857 | -0.0586 |
| trained | 200 | 0.902 | -0.0906 |
| untrained | 280 | 0.944 | -0.0358 |

L_proxy: Spearman S_0/S_1 = 0.575,

## 9. Mang kết quả về repo

Tải thư mục `MyDrive/DACNTT_smoke/results/` về và chép vào `experiments/week2September/manual/reports/task5/smoke_test/` trong repo. **Không** chép thư mục `adapters/` vào repo (trọng số model không được commit).

Sau đó viết `smoke_test_report.md` và cập nhật `task5_decisions.md`, config (v2 nếu đổi tham số), SHA256, changelog theo `Instruction_0.1.md` mục 9.

In [30]:
!ls -la "{RESULTS}"

total 1746
drwx------ 2 root root   4096 Oct 19 11:38 .
drwx------ 5 root root   4096 Oct 19 08:16 ..
-rw------- 1 root root  15856 Oct 19 10:09 environment_colab.txt
-rw------- 1 root root 473945 Oct 19 08:54 eval_S0_dev.jsonl
-rw------- 1 root root    922 Oct 19 10:09 eval_S0_dev.summary.json
-rw------- 1 root root 435037 Oct 19 10:41 eval_S1_dev_seed13.jsonl
-rw------- 1 root root   1001 Oct 19 10:41 eval_S1_dev_seed13.summary.json
-rw------- 1 root root 502063 Oct 19 11:18 eval_S1_dev_seed42.jsonl
-rw------- 1 root root   1004 Oct 19 11:18 eval_S1_dev_seed42.summary.json
-rw------- 1 root root 100886 Oct 19 11:25 nll_pool_S0.jsonl
-rw------- 1 root root    530 Oct 19 11:25 nll_pool_S0.summary.json
-rw------- 1 root root 104422 Oct 19 11:32 nll_pool_S1_seed13.jsonl
-rw------- 1 root root    609 Oct 19 11:32 nll_pool_S1_seed13.summary.json
-rw------- 1 root root 104440 Oct 19 11:38 nll_pool_S1_seed42.jsonl
-rw------- 1 root root    609 Oct 19 11:38 nll_pool_S1_seed42.summary.json
-rw